# ML-03: Frame Your Lane as an ML Task

**Lane:** CTR / Engagement Opportunity Scoring, continuing ML-02.

Results below are computed from the approved starter dataset.

## 1. My lane as an ML task

This is a **ranking/scoring** problem: which visible pages should a content/search specialist investigate first? The output is a limited review queue, with an explainable position-adjusted CTR gap and impression volume. Engagement is supporting context, not an automatic edit trigger.

For a content specialist deciding where to spend review time, we build a ranked queue from observed search metrics. A false positive wastes review time and could encourage an unnecessary edit; a false negative misses a useful review opportunity. The output supports checking search intent, titles/snippets and the page opening, or deciding to leave a page unchanged. It does not prescribe edits or promise additional clicks.

## 2. Target or proxy

The measured quantity for a later expected-CTR model is **observed CTR**, calculated as `100 * clicks_90d / impressions_90d`, in percentage points. It is not a binary label generated from our own opportunity rule.

For this descriptive prototype, reference CTR is the median within a position band, estimated only from reference clients. The queue score is `max(reference_ctr - observed_ctr, 0) / 100 * impressions_90d`. This is a **defined decision-support score**, not an observed outcome and not a target on which to train a model and claim independent validation.

The starter snapshot cannot demonstrate that a review or edit will improve future CTR. A forward-looking model needs features before a cutoff and CTR measured after it; 90-day totals and average position overlap recent 30-day outcomes and cannot be used as past-only predictors of those outcomes. Neither trend fields nor IDs belong in a predictive feature set.

## 3. Success metric

**Primary eventual decision metric:** precision@10, the proportion of the top ten candidates an independent human review judges actionable. A provisional acceptance target is at least 7/10 actionable candidates and better precision@10 than an impressions-only queue, using the same review rubric and budget. This threshold is a proposed operational target, not a measured result. Human labels are not available in the starter data, so precision@10 is not yet computable.

**Computable framing diagnostic:** impression-weighted MAE of the reference CTR on held-out clients, in percentage points. Compare a position-band reference with one global median reference. Lower error checks whether position conditioning adds descriptive value for unseen clients; it does not establish review quality or future uplift. This fixed split is for the framing exercise, not a tuned model evaluation.

## 4. Unit of analysis and real data

One input row is one pseudonymized content item with trailing-90-day metrics. Keep the ML-02 eligibility rule: at least 500 impressions and average position from 1 through 20. Position zero means missing, not first place. The printout below uses aggregate summaries; the five-row dataframe omits source IDs and all unnecessary fields.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data/raw/content_refresh_anonymized.csv').is_file())
data_path = ROOT / 'data/raw/content_refresh_anonymized.csv'
raw = pd.read_csv(data_path)
assert raw.shape == (30000, 44), 'Unexpected starter release shape'
assert raw['content_id'].is_unique, 'One row per content item required'
assert raw['client_id'].notna().all()
lane = raw.loc[(raw['impressions_90d'] >= 500)
               & raw['avg_position'].between(1, 20)].copy()
assert len(lane) > 0
assert lane['clicks_90d'].between(0, lane['impressions_90d']).all()
lane['observed_ctr_pct'] = 100 * lane['clicks_90d'] / lane['impressions_90d']
lane['position_band'] = pd.cut(lane['avg_position'], bins=[0, 3, 10, 20],
                               labels=['1-3', '>3-10', '>10-20'])
assert lane['position_band'].notna().all()
display(pd.DataFrame({
    'measure': ['starter rows', 'starter clients', 'eligible pages',
                'eligible percent', 'position zero rows excluded',
                'max difference from rounded supplied CTR (pp)'],
    'value': [len(raw), raw['client_id'].nunique(), len(lane),
              round(100 * len(lane) / len(raw), 2),
              int(raw['avg_position'].eq(0).sum()),
              round((lane['observed_ctr_pct'] - lane['ctr']).abs().max(), 6)]
}))
display(lane[['impressions_90d', 'clicks_90d', 'avg_position',
              'observed_ctr_pct', 'position_band']].head(5).reset_index(drop=True))


,measure,value
0,starter rows,30000.000
1,starter clients,32.000
2,eligible pages,12009.000
3,eligible percent,40.030
4,position zero rows excluded,1205.000
5,max difference from rounded supplied CTR (pp),0.005


,impressions_90d,clicks_90d,avg_position,observed_ctr_pct,position_band
0,3803,29,10.6,0.762556,>10-20
1,11751,58,6.2,0.493575,>3-10
2,3970,1,8.5,0.025189,>3-10
3,1240,2,4.9,0.161290,>3-10
4,20919,324,2.2,1.548831,1-3


### Reference and held-out clients

The split is deterministic: order client pseudonyms by a SHA-256 hash with fixed salt `ml03-v1`, hold out the first 20% (rounded up), and use the remaining clients for reference medians. IDs determine grouping only. All rows from a client stay together. Unseen position bands fall back to the reference global median. A positive gap is a queue proxy, not a confirmed problem.

In [2]:
clients = sorted(lane['client_id'].unique(),
                 key=lambda x: hashlib.sha256(('ml03-v1:' + str(x)).encode()).hexdigest())
n_holdout = max(1, int(np.ceil(len(clients) * 0.20)))
heldout_clients = set(clients[:n_holdout])
reference = lane.loc[~lane['client_id'].isin(heldout_clients)].copy()
evaluation = lane.loc[lane['client_id'].isin(heldout_clients)].copy()
assert not set(reference['client_id']) & set(evaluation['client_id'])
assert len(reference) and len(evaluation)
global_ctr = reference['observed_ctr_pct'].median()
band_ctr = reference.groupby('position_band', observed=True)['observed_ctr_pct'].median()
evaluation['reference_ctr_pct'] = (evaluation['position_band'].astype('string')
    .map(band_ctr).astype(float).fillna(global_ctr))
evaluation['ctr_gap_pp'] = (evaluation['reference_ctr_pct']
                          - evaluation['observed_ctr_pct']).clip(lower=0)
evaluation['directional_click_gap'] = (evaluation['ctr_gap_pp'] / 100
                                      * evaluation['impressions_90d'])
evaluation['future_review_actionable'] = pd.array([pd.NA] * len(evaluation), dtype='boolean')
def weighted_mae(predictions):
    return np.average(np.abs(evaluation['observed_ctr_pct'] - predictions),
                      weights=evaluation['impressions_90d'])
metrics = pd.DataFrame({
    'reference': ['global median', 'position-band median'],
    'heldout_impression_weighted_mae_pp': [weighted_mae(global_ctr),
        weighted_mae(evaluation['reference_ctr_pct'])]
})
display(pd.DataFrame({'partition': ['reference', 'held-out'],
                      'pages': [len(reference), len(evaluation)],
                      'clients': [reference['client_id'].nunique(),
                                  evaluation['client_id'].nunique()]}))
display(metrics.round(4))
change = 100 * (metrics.iloc[0, 1] - metrics.iloc[1, 1]) / metrics.iloc[0, 1]
print(f'Position conditioning changes weighted MAE by {change:.2f}% improvement (negative = worse).')
print('Descriptive client-holdout diagnostic only; no forward outcome or intervention validation.')


,partition,pages,clients
0,reference,10830,22
1,held-out,1179,6


,reference,heldout_impression_weighted_mae_pp
0,global median,0.4520
1,position-band median,0.4457


Position conditioning changes weighted MAE by 1.40% improvement (negative = worse).
Descriptive client-holdout diagnostic only; no forward outcome or intervention validation.


### What the target and queue columns look like

The first five held-out rows illustrate the observed target, reference and computed proxy. `future_review_actionable` is deliberately missing: there has been no independent human review. This is not a fabricated label. Only aggregate queue statistics are displayed; the complete queue stays in notebook memory.

In [3]:
display(evaluation[['observed_ctr_pct', 'reference_ctr_pct', 'ctr_gap_pp',
                    'directional_click_gap', 'future_review_actionable']]
        .head(5).reset_index(drop=True).round(4))
queue = evaluation.loc[evaluation['directional_click_gap'] > 0].sort_values(
    ['directional_click_gap', 'impressions_90d'], ascending=False, kind='stable')
display(pd.DataFrame({
    'measure': ['held-out positive-gap candidates', 'top-10 candidates available',
                'top-10 median gap (percentage points)', 'human-reviewed candidates'],
    'value': [len(queue), len(queue.head(10)),
              round(queue.head(10)['ctr_gap_pp'].median(), 4),
              int(evaluation['future_review_actionable'].notna().sum())]
}))
assert evaluation['future_review_actionable'].isna().all()
assert evaluation['directional_click_gap'].ge(0).all()
print('precision@10: not measured; independent actionability labels required.')


,observed_ctr_pct,reference_ctr_pct,ctr_gap_pp,directional_click_gap,future_review_actionable
0,0.0000,0.1679,0.1679,1.1605,<NA>
1,0.2315,0.1679,0.0000,0.0000,<NA>
2,0.0398,0.2306,0.1907,4.7873,<NA>
3,0.5309,0.2306,0.0000,0.0000,<NA>
4,0.0449,0.2306,0.1857,4.1394,<NA>


,measure,value
0,held-out positive-gap candidates,461.0000
1,top-10 candidates available,10.0000
2,top-10 median gap (percentage points),0.1708
3,human-reviewed candidates,0.0000


precision@10: not measured; independent actionability labels required.


## 5. Why data/ML may beat a fixed rule

A universal low-CTR threshold ignores that a page near position 2 and a page near position 18 face different conditions. A position-conditioned reference offers a transparent first comparison. Impression volume gives the review queue a practical scale, but it may favor large pages or clients; report client coverage and review a capped-per-client alternative later.

There is **no claim that ML already beats this baseline**. Query intent, device, country, brand demand and search-result features are potential confounders; most are not available in the starter slice. A more flexible model earns its place only after a leakage-safe comparison improves the predeclared metric. If it does not, keep the simple reference or use a dashboard. Broad position bands, small reference groups and the impression threshold also require sensitivity checks in later assignments.

## Self-check and next step

- [x] Task, decision, actor and error costs are explicit.
- [x] Measured CTR is distinguished from a rule-defined score and an unobserved future label.
- [x] Eligibility and one-row-per-content grain are checked in code.
- [x] Reference medians use separate clients from the diagnostic evaluation.
- [x] Only minimal example columns and aggregate metrics are displayed; no client names, URLs or raw queries.
- [x] Claims remain descriptive and decision-support oriented.
- [ ] Intern has reviewed and can explain the framing.
- [x] Notebook executed from top to bottom with no errors.

Commit and portal submission status are tracked in GitHub and the internship portal.

**Reproduce:** use Python with pandas, numpy, nbformat, nbclient and ipykernel installed, supply the approved starter CSV at `data/raw/content_refresh_anonymized.csv`, and run this notebook from top to bottom. No network request or model API call is made by the notebook.

**Sources:** local `work/notebooks/w01_research_question.ipynb`, `docs/data-dictionary.md`, `DATA_USE.md`, `skills/framing-ml-problems/SKILL.md` and `skills/flyrank/flyrank-data/SKILL.md`.